In this tutorial, we will use synthetic datasets generated from a subset of the 1000 Genomes Project consisting of 10,000 SNPs from chromosome 15. The datasets required for this tutorial are available in the git repository. 

### Data input

PRISM-G evaluates the privacy risk of synthetic genomic datasets by comparing them against the corresponding real genomic cohort. In addition, it uses two reference datasets, referred to as anchors, to calibrate the privacy score:

- A **safe reference dataset**, which preserves allele frequencies while removing the correlation structure between variants.
- A **leaky reference dataset**, which intentionally replicates samples from the training data.

These anchors define the calibration parameters of the PRISM-G score, representing the expected privacy risk under the safe and leaky reference scenarios, respectively. More information about these reference datasets is provided in **Calculating the Privacy Risk Score**. Instructions for generating the reference anchors for your own genomic cohort are available in **Generating the Reference Anchors**.

To import genotype data, PRISM-G reads VCF files using the `load_vcf()` function from the `vcf_reader` module. Input VCF files may be provided either uncompressed (`.vcf`) or compressed (`.vcf.gz`). **Note:** For accurate results, the VCF file must be phased and previously imputed.

If you wish to analyze a subset of SNPs, you must also provide a **legend file** describing the selected variants. The legend file should contain the following columns:

- `id`: SNP identifier
- `pos`: Chromosomal position
- `a0`: Reference allele
- `a1`: Alternate allele

The legend file ensures that the same set of variants is extracted and compared across all real and synthetic datasets.

In [1]:
import numpy as np
import pandas as pd

from prismg.io import vcf_reader as vcfio

In [2]:
# Real dataset
REAL_VCF = "../example_data/1000G_10K_SNP_chr15.vcf.gz"
# SNP Positions
LEGEND   = "../example_data/10K_SNP.legend"

# Synthetic datasets
GAN_VCF  = "../example_data/GAN_10K_SNP_chr15.vcf.gz"
RBM_VCF  = "../example_data/RBM_10K_SNP_chr15.vcf.gz"
GENOMATOR_H1_VCF = "../example_data/GENOMATOR_10K_chr15.vcf.gz"
GENOMATOR_H10_VCF = "../example_data/GENOMATOR_10K_H10_chr15.vcf.gz"
GENOMATOR_H50_VCF = "../example_data/GENOMATOR_10K_H50_chr15.vcf.gz"

# Calibrators
LEAKY_VCF = "../example_data/LEAKY_chr15.vcf.gz"
SAFE_VCF = "../example_data/SAFE_chr15.vcf.gz"

The `load_vcf()` function will return three objects: a list of the samples in the data, a list of the genomic positions of the variants, and the genotype matrix (dosage values).

In [3]:
legend_keys = vcfio.load_snp_legend_pos_keys(LEGEND)
real_samples, real_meta, real_G = vcfio.load_vcf(REAL_VCF, keep_pos=legend_keys)

gan_samples,  gan_meta,  gan_G  = vcfio.load_vcf(GAN_VCF,  keep_pos=legend_keys)
rbm_samples,  rbm_meta,  rbm_G  = vcfio.load_vcf(RBM_VCF,  keep_pos=legend_keys)
genom_h1_samples,  genom_h1_meta,  genom_h1_G  = vcfio.load_vcf(GENOMATOR_H1_VCF,  keep_pos=legend_keys)
genom_h10_samples,  genom_h10_meta,  genom_h10_G  = vcfio.load_vcf(GENOMATOR_H10_VCF,  keep_pos=legend_keys)
genom_h50_samples,  genom_h50_meta,  genom_h50_G  = vcfio.load_vcf(GENOMATOR_H50_VCF,  keep_pos=legend_keys)

leaky_samples,  leaky_meta,  leaky_G  = vcfio.load_vcf(LEAKY_VCF,  keep_pos=legend_keys)
safe_samples,  safe_meta,  safe_G  = vcfio.load_vcf(SAFE_VCF,  keep_pos=legend_keys)

Before estimating the PRISM-G components, the real dataset must be split into training and holdout sets. The training set is used to fit the sub-metrics that comprise each PRISM-G component, while the holdout set serves as an independent validation set for computing the final privacy metrics.

In [4]:
RANDOM_SEED   = 123
TRAIN_FRAC    = 0.8

rng = np.random.RandomState(RANDOM_SEED)
idx = np.arange(len(real_samples))
rng.shuffle(idx)
cut = int(len(idx) * TRAIN_FRAC)
tr_idx, ho_idx = idx[:cut], idx[cut:]

R_tr_samples = [real_samples[i] for i in tr_idx]
R_ho_samples = [real_samples[i] for i in ho_idx]
G_tr, G_ho   = real_G[tr_idx, :], real_G[ho_idx, :]

print(f"(TRAIN={len(R_tr_samples)}, HOLDOUT={len(R_ho_samples)})")

(TRAIN=2003, HOLDOUT=501)


### PRISM-G sub-metrics and components

Each PRISM-G component is estimated from a collection of complementary **sub-metrics** derived from similarity measurements, population genetic statistics, and membership inference attack diagnostics. These sub-metrics operate on different representations of the genotype data, including genetic distances, genomic relationship matrices (GRMs), and rare variant burden scores.

The final value of each component is obtained by aggregating its sub-metrics using one of three functions:

- **Maximum** (`max`) (default): reports the largest sub-metric value and represents a worst-case privacy risk assessment.
- **Mean** (`mean`): averages the contribution of all sub-metrics.
- **Median** (`median`): provides a robust estimate that is less sensitive to individual extreme values.

**Note:** The choice of aggregation function can influence both the individual component scores and the final **PRISM-G privacy score**.

We recommend using the default **`max`** aggregation when a single high-risk signal should be considered sufficient evidence of privacy leakage. 

Conversely, if multiple sub-metrics provide moderately different but consistent estimates, the **`mean`** or **`median`** may provide a more representative summary of the overall component, avoiding an overly conservative estimate based solely on the highest value.

Because the component values are subsequently aggregated into the final PRISM-G score, changing the aggregation scheme may alter the resulting privacy ranking of synthetic datasets. When comparing multiple datasets or generative models, the same aggregation strategy should therefore be used consistently across all analyses.

#### Proximity Leakage Index (PLI)

The PLI componenyt evaluates whether synthetic genomes lie unusually close to real individuals in genetic coordinate space. Genotypes are first embedded using **Principal Component Analysis (PCA)**, and distances between real and synthetic samples are computed using the nearest-neighbor Euclidean distance in the principal component space.

The PLI component is estimated from two complementary sub-metrics:

- **Quantile Proximity Ratio ($r_P$):** Evaluates whether the smallest distances between real and synthetic samples fall below the expected distribution of distances among real individuals.
- **Adversarial Proximity ($r_A$):** Measures whether holdout individuals are closer to synthetic samples than to other real individuals.

The PLI component is computed using the `compute_pli()` function from the `metrics.pli` module in PRISM-G. The function requires the genotype matrices for the real, synthetic, and holdout datasets as input, together with the number of principal components to compute and the quantile threshocld ($Q$) used to estimate the $r_P$ sub-metric.

In [5]:
from prismg.metrics.pli import compute_pli

# Parameters for PLI 
N_COMPONENTS  = 10
Q_LOWER       = 0.01

_pli_kwargs = dict(
    n_components = N_COMPONENTS,
    random_seed = RANDOM_SEED,
    q            = Q_LOWER,
)

_GENERATORS = [
    ("GAN",           gan_G),
    ("RBM",           rbm_G),
    ("Genomator_H1",  genom_h1_G),
    ("Genomator_H10", genom_h10_G),
    ("Genomator_H50", genom_h50_G),
    ("Leaky Copycat", leaky_G),
    ("Safe binomial", safe_G),
]

_SCORE_KEYS = ("r_p", "r_A", "PLI")

pli_df = pd.DataFrame([
    {"dataset": label, **{k: compute_pli(G_tr, G_ho, gen, **_pli_kwargs)[k] for k in _SCORE_KEYS}}
    for label, gen in _GENERATORS
]).set_index("dataset").round(4)

display(pli_df)

,r_p,r_A,PLI
dataset,,,
GAN,0.0000,0.3014,0.3014
RBM,0.0000,0.3453,0.3453
Genomator_H1,0.8087,0.8283,0.8283
Genomator_H10,0.5878,0.8084,0.8084
Genomator_H50,0.2621,0.7325,0.7325
Leaky Copycat,1.0000,0.8523,1.0000
Safe binomial,0.0000,0.0000,0.0000


#### Kinship Replay Index (KRI)

The KRI component evaluates whether synthetic genomes captures family structure patterns. Genotype matrix is transformed into a GRM to quantify pairwise relatedness among individuals. In addition, local haplotype patterns are used to identify repeated genotype segments as indication of shared ancestry.

The KRI component is estimated from four complementary sub-metrics:

- **Replay ($r_\mathrm{replay}$):** Evaluates whether the distribution of close-kin relationships observed in the real dataset is reproduced in the synthetic cohort.
- **Internal Kinship Excess ($r_{\mathrm{IKE}}$):** Measures whether synthetic samples exhibit elevated mutual relatedness relative to the real data by comparing the fraction of kinship values exceeding predefined thresholds.
- **Micro-haplotype Collisons ($r_\mathrm{HAP}$):** Detects repeated short haplotype segments shared across synthetic individuals.
- **Spectral Inflation ($r_\mathrm{SPEC}$):** Identifies concentration of relatedness structure through inflation of the leading eigenvalue of the GRM.

The KRI component is computed using the `compute_kri()` function from the `metrics.kri` module. The function accepts the following parameters:

- `theta`: Kinship threshold used by the Replay metric. Pairs with a kinship coefficient greater than this threshold are considered close relatives in the GRM matrix. Default: `theta = 0.125`.
- `replay_upper`: Upper clipping value for the close-kin distribution used to define the kinship histogram bins. Default: `replay_upper = 0.5`.
- `replay_n_bins`: Number of histogram bins between `theta` and `replay_upper` used to estimate the Jensen–Shannon divergence. Default: `replay_n_bins = 25`.
- `ike_thetas`: Kinship thresholds evaluated by the IKE sub-metric in the GRM. Default: `ike_thetas = (0.1, 0.125, 0.25)`.
- `var_chr`: Chromosome label for each SNP, used by the HAP sub-metric.
- `window_k`: Number of consecutive SNPs in each sliding window used to compute micro-haplotype collisions. Default: `window_k = 8`.
- `stride`: Step size between consecutive sliding windows. Default: `stride = 4`.
- `min_poly`: Minimum number of polymorphic SNPs required for a window to be included in the micro-haplotype collision analysis. Default: `min_poly = 6`.
- `n_boot`: Number of bootstrap replicates used to estimate the KRI sub-metrics. Default: `n_boot = 100`.

**Note:** Estimating the KRI component is computationally demanding. For the purpose of this tutorial, we show the results for `n_boot = 10`.

In [ ]:
from prismg.metrics.kri import compute_kri

# Parameters for KRI
THETA_REPLAY = 0.125
REPLAY_N_BINS = 25
REPLAY_UPPER = 0.5
IKE_THETAS   = (0.1, 0.125, 0.25)
WINDOW = 8
STRIDE = 4
MIN_POLY = 6
N_BOOT       = 10

var_chr = [chrom for chrom in real_meta]

_kri_kwargs = dict(
    theta        = THETA_REPLAY,
    replay_n_bins= REPLAY_N_BINS,
    replay_upper = REPLAY_UPPER,
    ike_thetas   = IKE_THETAS,
    window_k = WINDOW,
    stride = STRIDE,
    min_poly = MIN_POLY,
    n_boot       = N_BOOT,
    random_seed         = RANDOM_SEED
)

_SCORE_KEYS = ("r_replay", "r_IKE", "r_HAP", "r_SPEC", "KRI")

kri_df = pd.DataFrame([
    {"dataset": label, **{k: compute_kri(G_tr, G_ho, gen, var_chr, **_kri_kwargs)[k] for k in _SCORE_KEYS}}
    for label, gen in _GENERATORS
]).set_index("dataset").round(4)

display(kri_df)

,r_replay,r_IKE,r_HAP,r_SPEC,KRI
dataset,,,,,
GAN,0.0000,0.2655,0.0,0.4354,0.4354
RBM,0.0000,0.0000,0.0,0.0000,0.0000
Genomator_H1,0.5532,0.0000,0.0,0.7504,0.7504
Genomator_H10,0.6649,0.0188,0.0,0.7889,0.7889
Genomator_H50,0.6572,0.0557,0.0,0.8134,0.8134
Leaky Copycat,0.0000,0.0411,0.0,0.2060,0.2060
Safe binomial,0.0000,0.0000,0.0,0.0000,0.0000


#### Trait-linked leakage index

The TLI component evaluates whether distinctive rare variant patterns are preserved in synthetic genomes and whether rare variant burden scores can be used to re-identify individuals from the training dataset.

The TLI component is estimated from two complementary sub-metrics:

- **Membership Inference Signal ($r_{\mathrm{MIA}}$):** Evaluates whether individuals used to train the generative model can be distinguished from holdout individuals using a nearest-neighbor membership classifier based on rare variant burden scores.
- **Rare Variant Uniqueness ($r_{\mathrm{uniq}}$):** Measures whether uncommon variants are preserved more frequently in the synthetic dataset than expected.

The TLI component is computed using the `compute_tli()` function from the `metrics.tli` module. The function requires the genotype matrices for the real, synthetic, and holdout datasets, together with the chromosome labels for each variant. It also accepts the following parameters:

- `maf_thresh`: Minor allele frequency (MAF) threshold below which a variant is considered rare. Default: `maf_thresh = 1e-3`.
- `k_minor`: Minimum minor allele count required for a variant to be included in the analysis. Together with `maf_thresh`, this parameter defines the effective lower bound on the MAF. Default: `k_minor = 0`.
- `min_train_calls_frac`: Minimum fraction of training samples required to have a non-missing genotype call for a variant to be included in the analysis. Default: `min_train_calls_frac = 0.8`.

In [7]:
from prismg.metrics.tli import compute_tli

# Parameters for TLI
RARE_MAF = 1e-3
K_MINOR = 0

_tli_kwargs = dict(
    maf_thresh = RARE_MAF,
    k_minor = K_MINOR,
    min_train_calls_frac = TRAIN_FRAC,
)

_SCORE_KEYS = ("r_mia", "r_uniq", "TLI")

tli_df = pd.DataFrame([
    {"dataset": label, **{k: compute_tli(G_tr, G_ho, gen, var_chr, **_tli_kwargs)[k] for k in _SCORE_KEYS}}
    for label, gen in _GENERATORS
]).set_index("dataset").round(4)

display(tli_df)

,r_mia,r_uniq,TLI
dataset,,,
GAN,0.0000,0.2713,0.2713
RBM,0.0005,0.7944,0.7944
Genomator_H1,0.0390,0.0000,0.0390
Genomator_H10,0.0238,0.0000,0.0238
Genomator_H50,0.0349,0.0000,0.0349
Leaky Copycat,0.0000,1.0000,1.0000
Safe binomial,0.0121,0.0000,0.0121


In the next section, we describe how to estimate the aggregated privacy score with the obtained PRISM-G component measurements.